In [ ]:
import sienna
from model import SchemaIntegrationRuns
from evaluation import SchemaIntegrationEvaluation
import os

### Workflow Evaluation

In [ ]:
for log_file_name in os.listdir("logs/"):
    if f"{log_file_name.split('.json')[0]}_evaluation.json" not in os.listdir("evaluation_json/"):
        try:
            schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=f"logs/{log_file_name.split('.json')[0]}")
            schema_evaluator.run_evaluation()
            schema_evaluator.save_results()
        except Exception as e:
            print(f"Error processing {log_file_name}: {e}")

### COMA evaluation

In [ ]:
# COMA results evaluation
json_results = {}
benchmarks = ["Real Benchmark", "SINT-Benchmark"]
# Read COMA results
for benchmark in benchmarks:
    json_results[benchmark] = {}
    for coma_result_file in os.listdir(f"coma_results/{benchmark}/"):
        if coma_result_file.endswith("results.json"):
            coma_results = sienna.load(f"coma_results/{benchmark}/{coma_result_file}")
            folder_name = coma_result_file.replace("_coma_results.json", "")
            json_results[benchmark][folder_name] = {}
            gt_mappings = sienna.load(f"data/gt/{benchmark}/{folder_name}/{folder_name}_gt_mappings_with_values.json")

            # GT combinations
            true_combinations = set()
            for group in gt_mappings["grouped_attributes"].values():
                for tabattr in group:
                    for table, attribute in tabattr.items():
                        # Inner loop
                        for other_tabattr in group:
                            for other_table, other_attribute in other_tabattr.items():
                                true_combinations.add(tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])))

            
            for matcher in coma_results["matchers_mapped"]:
                # Predicted combinations
                combinations = []
                for matches in coma_results["matchers_mapped"][matcher]:
                    for table_match in matches:
                        combinations.append(tuple(sorted([f"{table_match[0][0]}.{table_match[0][1]}", f"{table_match[1][0]}.{table_match[1][1]}"])))

                tps = 0
                fps = 0
                fns = 0
                for combination in combinations:
                    if combination in true_combinations:
                        tps += 1
                    else:
                        fps += 1
                for combination in true_combinations:
                    if combination not in combinations:
                        fns += 1

                json_results[benchmark][folder_name][matcher] = {
                    "true_combinations": len(true_combinations),
                    "predicted_combinations": len(combinations),
                    "true_positives": tps,
                    "false_positives": fps,
                    "false_negatives": fns,
                    "precision": tps/(tps+fps) if (tps+fps) > 0 else 0,
                    "recall": tps/(tps+fns) if (tps+fns) > 0 else 0,
                    "f1_score": 2*tps/(2*tps + fps + fns) if (2*tps + fps + fns) > 0 else 0
                }

# Calculate average metrics across use cases and matchers for each benchmark and calculate overall average metrics
avg_metrics = { benchmark: {matcher: {f"avg_{metric}": sum([json_results[benchmark][use_case][matcher][metric] for use_case in json_results[benchmark]])/len(json_results[benchmark]) for metric in ["precision", "recall", "f1_score"]} for matcher in ["Coma", "Coma-instances", "Coma-instances-schema"]} for benchmark in benchmarks}

for benchmark in benchmarks:
    json_results[benchmark]["average_results"] = avg_metrics[benchmark]

sienna.save(json_results, "results_coma.json")

### SI-LLM evaluation

In [ ]:
# SI-LLM evaluation
benchmark = "Real Benchmark"
# benchmark = "SINT-Benchmark"
json_results = {}
for file_name in os.listdir("si-llm/"):
    # if ".json" in file_name and "Real" not in file_name:
    if ".json" in file_name and "Real" in file_name:
        log_file = sienna.load(f"si-llm/{file_name}")
        use_case = file_name.split("logs_")[1].split(f"_{benchmark}_GPT-5.2-SI-LLM")[0]
        print(use_case)
        # load ground truth
        gt_mappings = sienna.load(f"data/gt/{benchmark}/{use_case}/{use_case}_gt_mappings_with_values.json")

        # GT combinations
        true_combinations = set()
        for group in gt_mappings["grouped_attributes"].values():
            for tabattr in group:
                for table, attribute in tabattr.items():
                    # Inner loop
                    for other_tabattr in group:
                        for other_table, other_attribute in other_tabattr.items():
                            true_combinations.add(tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])))

        # Predicted combinations
        combinations = []
        for group in log_file["attribute_groups"]:
        # for group in schema_evaluator.predictions["0"]["parameters"]["attribute_groups_with_no_removals"]["all"]:
            for table, attributes in group.items():
                for attribute in attributes:
                    # Inner loop
                    for other_table, other_attributes in group.items():
                        for other_attribute in other_attributes:
                            if tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])) not in combinations:
                                combinations.append(tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])))

        
        # print("True combinations:", true_combinations)
        # print("Predicted combinations:", combinations)
        
        tps = 0
        fps = 0
        fns = 0
        for combination in combinations:
            if combination in true_combinations:
                tps += 1
            else:
                fps += 1
        for combination in true_combinations:
            if combination not in combinations:
                fns += 1

        json_results[use_case] = {
            "true_combinations": len(true_combinations),
            "predicted_combinations": len(combinations),
            "true_positives": tps,
            "false_positives": fps,
            "false_negatives": fns,
            "precision": tps/(tps+fps) if (tps+fps) > 0 else 0,
            "recall": tps/(tps+fns) if (tps+fns) > 0 else 0,
            "f1_score": 2*tps/(2*tps + fps + fns) if (2*tps + fps + fns) > 0 else 0
        }
            # except Exception as e:
            #     print(f"Error processing {use_case}: {e}")
                
# Calculate average metrics across use cases
f1s = [json_results[folder]["f1_score"] for folder in json_results]
p = [json_results[folder]["precision"] for folder in json_results]
r = [json_results[folder]["recall"] for folder in json_results]
average_f1 = sum(f1s) / len(f1s) if f1s else 0
average_precision = sum(p) / len(p) if p else 0
average_recall = sum(r) / len(r) if r else 0

json_results["average_results"] = {
    "average_f1": average_f1,
    "average_precision": average_precision,
    "average_recall": average_recall
}
sienna.save(json_results, f"results_like_alite_{benchmark}-gpt-5.2-si-llm.json")

### SMO (Schema Matching Operator) evaluation

In [ ]:
benchmark = "Real Benchmark"
# benchmark = "SINT-Benchmark"
json_results = {}
# Take the schema matching first runs and evaluate them as ALITE does
for file_name in os.listdir("logs/"):
    # if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" in file_name:
    if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" not in file_name:
    # if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" in file_name:
    # if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" not in file_name:
        log_file_name = "logs/" + file_name.replace("_evaluation.json", "").replace(".json","")
        schema_evaluator = SchemaIntegrationEvaluation(tables_path=f"data/selected-tables/{benchmark}", log_file_name=log_file_name)

        if schema_evaluator.run_info["sequence_of_phases"] == ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"]:
            schema_evaluator.run_evaluation()
            eval_results = schema_evaluator.eval_results
            use_case = schema_evaluator.run_info["folder_name"]

            # GT combinations
            true_combinations = set()
            for group in schema_evaluator.gt_mappings["all"]["grouped_attributes"].values():
                for tabattr in group:
                    for table, attribute in tabattr.items():
                        # Inner loop
                        for other_tabattr in group:
                            for other_table, other_attribute in other_tabattr.items():
                                true_combinations.add(tuple(sorted([f"table_{schema_evaluator.file_names_to_index[table]}.{attribute}", f"table_{schema_evaluator.file_names_to_index[other_table]}.{other_attribute}"])))


            # Predicted combinations
            combinations = []
            for group in schema_evaluator.predictions["0"]["parameters"]["attribute_groups"]["all"]:
                for table, attributes in group.items():
                    for attribute in attributes:
                        # Inner loop
                        for other_table, other_attributes in group.items():
                            for other_attribute in other_attributes:
                                if tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])) not in combinations:
                                    combinations.append(tuple(sorted([f"{table}.{attribute}", f"{other_table}.{other_attribute}"])))

            
            tps = 0
            fps = 0
            fns = 0
            for combination in combinations:
                if combination in true_combinations:
                    tps += 1
                else:
                    fps += 1
            for combination in true_combinations:
                if combination not in combinations:
                    fns += 1

            json_results[use_case] = {
                "true_combinations": len(true_combinations),
                "predicted_combinations": len(combinations),
                "true_positives": tps,
                "false_positives": fps,
                "false_negatives": fns,
                "precision": tps/(tps+fps) if (tps+fps) > 0 else 0,
                "recall": tps/(tps+fns) if (tps+fns) > 0 else 0,
                "f1_score": 2*tps/(2*tps + fps + fns) if (2*tps + fps + fns) > 0 else 0
            }
            
# Calculate average metrics across use cases
f1s = [json_results[folder]["f1_score"] for folder in json_results]
p = [json_results[folder]["precision"] for folder in json_results]
r = [json_results[folder]["recall"] for folder in json_results]
average_f1 = sum(f1s) / len(f1s) if f1s else 0
average_precision = sum(p) / len(p) if p else 0
average_recall = sum(r) / len(r) if r else 0

json_results["average_results"] = {
    "average_f1": average_f1,
    "average_precision": average_precision,
    "average_recall": average_recall
}
# sienna.save(json_results, f"results_like_alite_{benchmark}-gpt-5.2.json")
sienna.save(json_results, f"results_like_alite_{benchmark}-Qwen.json")